# Risk-Controlled Momentum: Day 1 Prototype

This notebook develops the pre-registered model sequentially. Each section should be understood and checked before proceeding.

## 1. Imports and configuration

### Concept before code

An **import** makes a library's tools available in the notebook. The short names are conventional aliases: `np` for NumPy, `pd` for pandas, `plt` for Matplotlib plotting, `sns` for seaborn and `yf` for yfinance. An alias changes only how we refer to a library; it does not change the library.

A Python **dictionary** stores labelled key-value pairs. Keeping every pre-registered choice in `LOCKED_CONFIG` separates research decisions from later calculations and makes accidental parameter changes easier to notice. Percentages are decimals in calculations: 10% is `0.10`. One basis point is one ten-thousandth, so 10 basis points is `10 / 10_000 = 0.001`.

The one-day `timing_lag_days` value records the rule that a return on day `t` can use information available no later than day `t-1`. It does not perform the lag yet; the lag will be implemented and checked in the signal and volatility sections.

The final two lines turn the dictionary into a one-column table for visual inspection. Nothing in this section downloads data or calculates a strategy result.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf

LOCKED_CONFIG = {
    "ticker": "SPY",
    "sample_start": "2007-01-01",
    "sample_end": "2025-12-31",
    "frequency": "daily",
    "price_field": "adjusted",
    "return_type": "simple",
    "momentum_lookback_days": 252,
    "volatility_window_days": 21,
    "annualisation_days": 252,
    "target_volatility": 0.10,
    "maximum_exposure": 1.5,
    "rebalancing": "daily",
    "transaction_cost_bps": 10,
    "transaction_cost_rate": 10 / 10_000,
    "cash_return": 0.0,
    "timing_lag_days": 1,
}

config_table = pd.Series(LOCKED_CONFIG, name="Locked Day 1 value").to_frame()
config_table

## 2. Download and validate adjusted price data

### Concept before code

A **DataFrame** is a labelled table with rows and columns. `yf.download(...)` returns a DataFrame containing daily market fields. A **Series** is one labelled column; after inspecting the download, we select only the adjusted SPY closing-price series needed by this model.

SPY pays distributions. An unadjusted close can fall when cash leaves the fund even though an investor received that cash. Because the locked specification requires an adjusted price, we explicitly set `auto_adjust=True`. In installed `yfinance 1.6.0`, this adjusts the OHLC fields and places the adjusted closing price in `Close`; a separate `Adj Close` column is therefore absent.

The provider treats `start` as inclusive and `end` as exclusive. We convert the locked end-date string to a pandas `Timestamp`, add a one-day `Timedelta`, and convert it back to a date string. Requesting an exclusive boundary of `2026-01-01` is how we include the locked final date `2025-12-31`; it does not extend the research sample.

The validation dictionary stores named True/False checks. We require a datetime index, increasing and unique dates, no missing selected prices, positive prices, and coverage contained within both ends of the locked sample. The seven-calendar-day tolerance recognises that 1 January or 31 December can be a weekend or market holiday; it is a data-coverage check, not a model parameter. If any check is False, `raise ValueError(...)` stops the notebook instead of allowing bad data to flow into the strategy.

In [ ]:
sample_start = pd.Timestamp(LOCKED_CONFIG["sample_start"])
sample_end = pd.Timestamp(LOCKED_CONFIG["sample_end"])
download_end_exclusive = (sample_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

raw_spy = yf.download(
    tickers=LOCKED_CONFIG["ticker"],
    start=LOCKED_CONFIG["sample_start"],
    end=download_end_exclusive,
    interval="1d",
    auto_adjust=True,
    actions=False,
    keepna=False,
    progress=False,
    multi_level_index=False,
)

if raw_spy.empty:
    raise RuntimeError("SPY download returned no observations.")

if "Close" not in raw_spy.columns:
    raise ValueError(f"Adjusted Close field not found. Returned columns: {raw_spy.columns.tolist()}")

spy_price = raw_spy["Close"].rename("adjusted_price")

validation_checks = {
    "datetime_index": isinstance(spy_price.index, pd.DatetimeIndex),
    "dates_in_increasing_order": spy_price.index.is_monotonic_increasing,
    "dates_are_unique": not spy_price.index.has_duplicates,
    "no_missing_prices": not spy_price.isna().any(),
    "all_prices_are_positive": spy_price.gt(0).all(),
    "starts_inside_locked_sample": spy_price.index.min() >= sample_start,
    "ends_inside_locked_sample": spy_price.index.max() <= sample_end,
    "covers_locked_sample_start": spy_price.index.min() <= sample_start + pd.Timedelta(days=7),
    "covers_locked_sample_end": spy_price.index.max() >= sample_end - pd.Timedelta(days=7),
}

failed_checks = [name for name, passed in validation_checks.items() if not passed]
if failed_checks:
    raise ValueError(f"SPY data validation failed: {failed_checks}")

data_summary = pd.Series(
    {
        "ticker": LOCKED_CONFIG["ticker"],
        "selected_field": spy_price.name,
        "first_observation": spy_price.index.min().date(),
        "last_observation": spy_price.index.max().date(),
        "observations": spy_price.size,
        "missing_prices": int(spy_price.isna().sum()),
        "download_end_exclusive": download_end_exclusive,
    },
    name="Validated value",
)

display(data_summary.to_frame())
display(pd.Series(validation_checks, name="Passed").to_frame())
display(spy_price.head(3).to_frame())
display(spy_price.tail(3).to_frame())

## 3. Calculate simple daily returns

### Concept before code

A **simple return** measures the fractional change from one adjusted closing price to the next: `return[t] = price[t] / price[t-1] - 1`. If price rises from 100 to 102, the simple return is `0.02`, which means 2%. If it falls from 100 to 98, the return is `-0.02`, or -2%. Returns are stored as decimals, not as numbers already multiplied by 100.

`pct_change()` applies this calculation between consecutive rows. Despite its name, pandas returns a fractional change: `0.02`, not `2`. We explicitly use `fill_method=None` so a missing price would not be silently filled before the calculation.

The first return is necessarily `NaN` (not a number) because the first in-sample price has no earlier in-sample price. We retain the same index as the price series, so the return labelled `2007-01-04` represents the movement from the close on `2007-01-03` to the close on `2007-01-04`.

Validation checks require identical price/return dates, exactly one missing value in the first position, no later missing or infinite values, and agreement between `pct_change()` and one return calculated directly from the locked formula.

In [ ]:
spy_return = spy_price.pct_change(fill_method=None).rename("asset_return")

manual_second_return = spy_price.iloc[1] / spy_price.iloc[0] - 1
usable_returns = spy_return.dropna()

return_validation_checks = {
    "same_index_as_prices": spy_return.index.equals(spy_price.index),
    "same_number_of_rows_as_prices": spy_return.size == spy_price.size,
    "first_return_is_missing": pd.isna(spy_return.iloc[0]),
    "exactly_one_missing_return": spy_return.isna().sum() == 1,
    "no_missing_returns_after_first": spy_return.iloc[1:].notna().all(),
    "all_usable_returns_are_finite": np.isfinite(usable_returns).all(),
    "second_return_matches_formula": np.isclose(spy_return.iloc[1], manual_second_return),
}

failed_return_checks = [
    name for name, passed in return_validation_checks.items() if not passed
]
if failed_return_checks:
    raise ValueError(f"SPY return validation failed: {failed_return_checks}")

return_summary = pd.Series(
    {
        "price_observations": spy_price.size,
        "usable_return_observations": usable_returns.size,
        "missing_returns": int(spy_return.isna().sum()),
        "first_usable_return_date": usable_returns.index.min().date(),
        "minimum_daily_return": usable_returns.min(),
        "maximum_daily_return": usable_returns.max(),
    },
    name="Validated value",
)

return_preview = pd.concat([spy_price, spy_return], axis=1).head(4)
manual_example = pd.Series(
    {
        "previous_adjusted_price": spy_price.iloc[0],
        "current_adjusted_price": spy_price.iloc[1],
        "manual_simple_return": manual_second_return,
        "pct_change_return": spy_return.iloc[1],
    },
    name=usable_returns.index[0].date(),
)

display(return_summary.to_frame())
display(pd.Series(return_validation_checks, name="Passed").to_frame())
display(return_preview)
display(manual_example.to_frame())

## 4. Build the buy-and-hold benchmark

### Concept before code

A **portfolio weight** describes the fraction of capital exposed to an asset. Buy and hold uses a constant SPY weight of `1.0`, meaning 100% exposure. Its daily return is therefore `1.0 * asset_return`, so it should match SPY's adjusted-price return on every usable date. This section builds a gross benchmark and does not introduce transaction costs.

A **wealth index** answers what one unit of starting capital would become. Wealth compounds multiplicatively: `wealth[t] = wealth[t-1] * (1 + return[t])`. The daily value `1 + return` is a growth factor; a 2% return produces a factor of `1.02`, while a -2% return produces `0.98`. `cumprod()` multiplies those growth factors through time.

The first benchmark return remains `NaN`, because it is still undefined. For the wealth calculation only, we set the first growth factor to `1.0` to represent initial wealth before any observed return. This does not reclassify the missing return as an observed zero return.

Because the returns came from one adjusted-price series, a correctly compounded buy-and-hold wealth index must equal `adjusted_price / first_adjusted_price`. That independent equivalence is our strongest benchmark check.

In [ ]:
buy_hold_exposure = pd.Series(1.0, index=spy_price.index, name="buy_hold_exposure")
buy_hold_return = (buy_hold_exposure * spy_return).rename("buy_hold_return")

buy_hold_growth = (1.0 + buy_hold_return).rename("buy_hold_growth")
buy_hold_growth.iloc[0] = 1.0
buy_hold_wealth = buy_hold_growth.cumprod().rename("buy_hold_wealth")
normalized_adjusted_price = (spy_price / spy_price.iloc[0]).rename(
    "normalized_adjusted_price"
)

buy_hold_validation_checks = {
    "exposure_is_always_one": buy_hold_exposure.eq(1.0).all(),
    "first_benchmark_return_is_missing": pd.isna(buy_hold_return.iloc[0]),
    "usable_returns_match_asset": np.allclose(
        buy_hold_return.iloc[1:], spy_return.iloc[1:]
    ),
    "initial_wealth_is_one": np.isclose(buy_hold_wealth.iloc[0], 1.0),
    "wealth_has_no_missing_values": buy_hold_wealth.notna().all(),
    "wealth_is_positive_and_finite": (
        buy_hold_wealth.gt(0).all() and np.isfinite(buy_hold_wealth).all()
    ),
    "wealth_matches_normalized_price": np.allclose(
        buy_hold_wealth, normalized_adjusted_price
    ),
}

failed_buy_hold_checks = [
    name for name, passed in buy_hold_validation_checks.items() if not passed
]
if failed_buy_hold_checks:
    raise ValueError(f"Buy-and-hold validation failed: {failed_buy_hold_checks}")

buy_hold_summary = pd.Series(
    {
        "constant_exposure": buy_hold_exposure.iloc[-1],
        "initial_wealth": buy_hold_wealth.iloc[0],
        "ending_wealth": buy_hold_wealth.iloc[-1],
        "cumulative_return": buy_hold_wealth.iloc[-1] - 1.0,
        "first_date": buy_hold_wealth.index.min().date(),
        "last_date": buy_hold_wealth.index.max().date(),
    },
    name="Validated value",
)

benchmark_preview = pd.concat(
    [spy_price, spy_return, buy_hold_exposure, buy_hold_return, buy_hold_wealth],
    axis=1,
).head(4)
wealth_equivalence_preview = pd.concat(
    [buy_hold_wealth, normalized_adjusted_price], axis=1
).tail(3)

display(buy_hold_summary.to_frame())
display(pd.Series(buy_hold_validation_checks, name="Passed").to_frame())
display(benchmark_preview)
display(wealth_equivalence_preview)

## 5. Build and lag the momentum signal

## 6. Estimate and lag rolling volatility

## 7. Apply volatility targeting and the leverage cap

## 8. Calculate turnover and transaction costs

## 9. Calculate performance metrics

## 10. Visualise preliminary results

## 11. Run sanity checks and record observations